# Mini-projet 1 : Méta-analyse des articles de recherche sur les LLM
**Thème choisi : l'efficacité et l'alignement des grands modèles de langage**

Ce rapport analyse quatre articles fondateurs qui abordent, sous des angles complémentaires, la question suivante : *comment rendre les LLM plus performants sans se contenter d'augmenter leur taille ?* Les articles couvrent trois leviers distincts mais liés : les lois d'échelle et l'allocation optimale du calcul (Chinchilla), l'ouverture et l'efficacité des données d'entraînement (LLaMA), l'efficacité architecturale au moment de l'inférence (Mistral 7B), et l'alignement par retour humain (InstructGPT). Ensemble, ils dessinent une évolution du champ : de "plus gros est meilleur" vers "mieux entraîné, mieux conçu et mieux aligné est meilleur".

## 1. Introduction

Les grands modèles de langage (LLM) sont des réseaux de neurones, généralement basés sur l'architecture Transformer, entraînés sur d'immenses corpus de texte pour prédire la suite d'une séquence. Depuis 2020, la taille de ces modèles a explosé (GPT-3, PaLM, Gopher, etc.), mais cette course à l'échelle a rapidement révélé ses limites : coûts d'entraînement et d'inférence prohibitifs, dépendance à des données propriétaires, et surtout un décalage persistant entre ce que le modèle sait prédire (le prochain mot) et ce que l'utilisateur attend réellement (une réponse utile et sûre).

L'objectif de cette méta-analyse est d'examiner comment la recherche récente a répondu à ce problème en explorant trois leviers d'amélioration alternatifs à la simple augmentation du nombre de paramètres : (1) l'allocation optimale du calcul entre taille du modèle et quantité de données d'entraînement, (2) la conception d'architectures plus efficaces au moment de l'inférence, et (3) l'alignement des modèles sur les intentions humaines via le retour humain. Le fil conducteur des quatre articles sélectionnés est donc **l'efficacité et l'alignement des LLM** : comment obtenir de meilleurs modèles avec moins de ressources et un comportement plus fiable, plutôt qu'en misant uniquement sur la taille brute.

Les quatre articles retenus sont :
1. *Training Compute-Optimal Large Language Models* (Chinchilla) - Hoffmann et al., 2022, NeurIPS / arXiv.
2. *LLaMA: Open and Efficient Foundation Language Models* - Touvron et al., 2023, arXiv (Meta AI).
3. *Mistral 7B* - Jiang et al., 2023, arXiv (Mistral AI).
4. *Training Language Models to Follow Instructions with Human Feedback* (InstructGPT) - Ouyang et al., 2022, NeurIPS / arXiv (OpenAI).

## 2. Résumés des articles

### 2.1 Chinchilla - Training Compute-Optimal Large Language Models
**Citation complète :** Hoffmann, J., Borgeaud, S., Mensch, A., Buchatskaya, E., Cai, T., Rutherford, E., de Las Casas, D., Hendricks, L. A., Welbl, J., Clark, A., Hennigan, T., Noland, E., Millican, K., van den Driessche, G., Damoc, B., Guy, A., Osindero, S., Simonyan, K., Elsen, E., Rae, J. W., Vinyals, O. & Sifre, L. (2022). *Training Compute-Optimal Large Language Models*. Advances in Neural Information Processing Systems (NeurIPS 2022) ; arXiv:2203.15556.

**Problème de recherche :** à budget de calcul fixe, quelle est la meilleure répartition entre la taille du modèle (nombre de paramètres) et la quantité de données d'entraînement (nombre de tokens) ? Les auteurs constatent que les grands modèles de l'époque (Gopher, GPT-3, Jurassic-1, Megatron-Turing NLG) sont systématiquement sous-entraînés : ils sont trop gros par rapport à la quantité de données qu'ils ont réellement vues.

**Solution proposée :** DeepMind entraîne plus de 400 modèles, de 70 millions à plus de 16 milliards de paramètres, sur des volumes de données allant de 5 à 500 milliards de tokens, afin d'établir empiriquement des lois d'échelle. Le résultat central est que la taille du modèle et le nombre de tokens d'entraînement doivent croître au même rythme : chaque doublement de la taille du modèle doit s'accompagner d'un doublement des données d'entraînement. Cette prédiction est validée en entraînant Chinchilla, un modèle de 70 milliards de paramètres sur 1,4 trillion de tokens, avec le même budget de calcul que Gopher (280 milliards de paramètres, environ 300 milliards de tokens).

**Principaux résultats :** Chinchilla surpasse de façon uniforme des modèles bien plus gros que lui - Gopher (280B), GPT-3 (175B), Jurassic-1 (178B) et Megatron-Turing NLG (530B) - sur un large éventail de tâches d'évaluation en aval. Il atteint notamment une précision moyenne de 67,5 % sur le benchmark MMLU, soit une amélioration de plus de 7 points par rapport à Gopher, tout en étant nettement moins coûteux à affiner et à utiliser en inférence.

**Données, architecture, métriques :** modèles Transformer denses classiques (même famille architecturale que Gopher) ; corpus d'entraînement de type MassiveText (texte web, livres, code) ; évaluation sur un large ensemble de benchmarks académiques dont MMLU, ainsi que des tâches de langage et de raisonnement en aval.

### 2.2 LLaMA - Open and Efficient Foundation Language Models
**Citation complète :** Touvron, H., Lavril, T., Izacard, G., Martinet, X., Lachaux, M.-A., Lacroix, T., Rozière, B., Goyal, N., Hambro, E., Azhar, F., Rodriguez, A., Joulin, A., Grave, E. & Lample, G. (2023). *LLaMA: Open and Efficient Foundation Language Models*. arXiv:2302.13971 (Meta AI).

**Problème de recherche :** les modèles les plus performants (GPT-3, PaLM, Chinchilla) reposent souvent sur des données d'entraînement propriétaires et inaccessibles, ce qui freine la recherche ouverte et la reproductibilité. La question posée est : peut-on atteindre un niveau de performance comparable en n'utilisant que des données publiques ?

**Solution proposée :** Meta AI entraîne une famille de modèles fondation, LLaMA, de 7 à 65 milliards de paramètres, exclusivement sur des sources de données publiques (CommonCrawl, C4, Github, Wikipédia, livres, arXiv, StackExchange), sur des volumes atteignant plusieurs trillions de tokens. L'article s'inscrit directement dans la continuité des lois d'échelle de Chinchilla, en appliquant le principe "plus de données, modèle plus petit" à un cadre entièrement public.

**Principaux résultats :** LLaMA-13B surpasse GPT-3 (175B) sur la plupart des benchmarks malgré une taille plus de dix fois inférieure, et LLaMA-65B est compétitif avec Chinchilla-70B et PaLM-540B. Ces résultats démontrent qu'il est possible d'atteindre l'état de l'art sans recourir à des données propriétaires, ce qui a ouvert la voie à tout un écosystème de modèles ouverts dérivés.

**Données, architecture, métriques :** architecture Transformer décodeur avec des améliorations inspirées de PaLM et GPTNeo (normalisation RMSNorm, activation SwiGLU, encodages positionnels rotatifs RoPE) ; corpus exclusivement public totalisant environ 1,4 trillion de tokens pour le plus grand modèle ; évaluation en zero-shot et few-shot sur des benchmarks de raisonnement de bon sens, de compréhension du langage (MMLU), de questions-réponses et de génération de code.

### 2.3 Mistral 7B
**Citation complète :** Jiang, A. Q., Sablayrolles, A., Mensch, A., Bamford, C., Chaplot, D. S., de Las Casas, D., Bressand, F., Lengyel, G., Lample, G., Saulnier, L., Renard Lavaud, L., Lachaux, M.-A., Stock, P., Le Scao, T., Lavril, T., Wang, T., Lacroix, T. & El Sayed, W. (2023). *Mistral 7B*. arXiv:2310.06825 (Mistral AI).

**Problème de recherche :** au-delà de la quantité de données d'entraînement, la conception même de l'architecture influence l'efficacité d'un modèle en inférence (vitesse, mémoire, gestion des séquences longues). Comment concevoir un modèle de taille modeste qui reste rapide à exécuter tout en rivalisant avec des modèles plus grands ?

**Solution proposée :** les auteurs introduisent Mistral 7B, un modèle de 7 milliards de paramètres qui combine deux mécanismes d'attention optimisés : l'attention par groupes de requêtes (Grouped-Query Attention, GQA), qui accélère l'inférence et réduit les besoins en mémoire lors du décodage, et l'attention à fenêtre glissante (Sliding Window Attention, SWA), qui permet de traiter des séquences de longueur arbitraire à coût de calcul réduit en limitant la portée directe de l'attention à une fenêtre de tokens tout en laissant l'information se propager sur plusieurs couches.

**Principaux résultats :** Mistral 7B surpasse Llama 2 13B sur l'ensemble des benchmarks évalués, et Llama 1 34B sur les tâches de raisonnement, de mathématiques et de génération de code, malgré un nombre de paramètres nettement inférieur. Une version affinée pour suivre des instructions, Mistral 7B - Instruct, dépasse également Llama 2 13B - Chat, à la fois sur des évaluations humaines et automatisées. Le modèle est publié sous licence Apache 2.0, entièrement permissive.

**Données, architecture, métriques :** architecture Transformer décodeur avec GQA et SWA (fenêtre de contexte glissante) ; données d'entraînement non détaillées publiquement dans l'article (contrairement à LLaMA) ; évaluation sur des benchmarks de raisonnement de bon sens, de compréhension, de mathématiques (type GSM8K) et de génération de code, comparée directement à Llama 1/2.

### 2.4 InstructGPT - Training Language Models to Follow Instructions with Human Feedback
**Citation complète :** Ouyang, L., Wu, J., Jiang, X., Almeida, D., Wainwright, C., Mishkin, P., Zhang, C., Agarwal, S., Slama, K., Ray, A., Schulman, J., Hilton, J., Kelton, F., Miller, L., Simens, M., Askell, A., Welinder, P., Christiano, P., Leike, J. & Lowe, R. (2022). *Training Language Models to Follow Instructions with Human Feedback*. Advances in Neural Information Processing Systems (NeurIPS 2022) ; arXiv:2203.02155.

**Problème de recherche :** l'objectif d'entraînement classique d'un LLM - prédire le mot suivant sur des pages web - est fondamentalement différent de l'objectif recherché par les utilisateurs : suivre une instruction de façon utile, honnête et sans danger. Cet écart, appelé "misalignment", conduit les modèles à produire des réponses non pertinentes, peu sûres ou peu fiables même lorsqu'ils sont très performants sur des benchmarks classiques.

**Solution proposée :** OpenAI applique l'apprentissage par renforcement à partir de retours humains (RLHF) à grande échelle sur GPT-3. Le processus se déroule en trois étapes : (1) un fine-tuning supervisé (SFT) sur des démonstrations humaines de bonnes réponses, (2) l'entraînement d'un modèle de récompense (Reward Model) à partir de comparaisons humaines entre plusieurs réponses générées, et (3) l'optimisation du modèle de langage par renforcement (PPO) pour maximiser cette récompense.

**Principaux résultats :** les modèles InstructGPT, y compris une version de seulement 1,3 milliard de paramètres, sont préférés par les évaluateurs humains à GPT-3 (175 milliards de paramètres) malgré un nombre de paramètres jusqu'à 100 fois inférieur. Les réponses InstructGPT sont préférées à celles de GPT-3 dans environ 71 % des comparaisons humaines. Le RLHF améliore aussi la véracité des réponses et réduit la génération de contenus toxiques, avec toutefois un léger coût de performance sur certaines tâches de traitement du langage classiques, que les auteurs nomment la "taxe d'alignement".

**Données, architecture, métriques :** modèle de base GPT-3 (Transformer décodeur, jusqu'à 175B de paramètres) ; jeux de données composés de démonstrations humaines et de comparaisons de préférences collectées auprès d'annotateurs recrutés par OpenAI, à partir de prompts soumis via l'API GPT-3 ; évaluation par préférence humaine directe (taux de préférence face à GPT-3), ainsi que sur des benchmarks de véracité (TruthfulQA) et de toxicité, en plus des benchmarks NLP publics classiques.

## 3. Analyse comparative

Les quatre articles partagent un objectif commun - améliorer les LLM sans se contenter d'augmenter leur taille - mais s'attaquent à des maillons différents de la chaîne de valeur d'un modèle : les données d'entraînement (Chinchilla, LLaMA), l'architecture d'inférence (Mistral 7B), et le comportement final du modèle (InstructGPT). Le tableau ci-dessous synthétise ces différences selon les axes demandés.

| Aspect | Chinchilla (2022) | LLaMA (2023) | Mistral 7B (2023) | InstructGPT (2022) |
| --- | --- | --- | --- | --- |
| **Objectif / problème** | Trouver la répartition optimale calcul/données/taille | Égaler l'état de l'art avec des données 100 % publiques | Maximiser l'efficacité d'inférence à taille de modèle réduite | Aligner le comportement du modèle sur l'intention humaine |
| **Levier d'amélioration** | Lois d'échelle (scaling laws) | Ouverture et qualité des données | Architecture (attention GQA + SWA) | Retour humain (RLHF) |
| **Architecture / innovation** | Transformer dense classique, focus sur le ratio taille/données | Transformer + RMSNorm, SwiGLU, RoPE | Transformer + Grouped-Query Attention et Sliding Window Attention | GPT-3 + pipeline SFT -> Reward Model -> PPO |
| **Stratégie d'entraînement / fine-tuning** | Pré-entraînement seul, comparaison à budget de calcul égal | Pré-entraînement seul sur corpus public uniquement | Pré-entraînement + variante Instruct affinée | Pré-entraînement (GPT-3) + SFT + RLHF (PPO) |
| **Benchmarks principaux** | MMLU et large batterie de tâches NLP en aval | MMLU, raisonnement de bon sens, code, QA | Raisonnement, mathématiques, code (comparaison directe à Llama 1/2) | Préférence humaine directe, TruthfulQA, toxicité |
| **Points forts** | Gains de performance importants sans coût de calcul supplémentaire | Performance état de l'art avec données et poids plus accessibles | Petit modèle qui bat des modèles 2 a 5x plus gros en inférence | Amélioration mesurable de l'utilité et de la sécurité perçues |
| **Limites** | Ne traite pas l'alignement ni l'efficacité d'inférence | Ne résout pas l'alignement ; licence d'usage encore restrictive à l'origine | N'aborde pas l'alignement comportemental | "Taxe d'alignement" : légère perte de performance sur certaines tâches ; dépendance à des annotateurs humains coûteux à grande échelle |
| **Reproductibilité** | Modèles et poids non publiés (recherche interne DeepMind) | Poids publiés (avec restrictions initiales), méthodo. et données détaillées | Poids et licence Apache 2.0 entièrement ouverts | Données de préférence humaines non publiées ; méthode documentée mais coûteuse à reproduire |

Un point de comparaison intéressant est la notion d'"efficacité", qui ne signifie pas la même chose selon l'article : pour Chinchilla, elle se mesure en performance par unité de calcul d'entraînement ; pour LLaMA, en performance par accès aux données (publiques plutôt que propriétaires) ; pour Mistral 7B, en performance par coût d'inférence (vitesse, mémoire) ; et pour InstructGPT, en performance perçue par l'utilisateur final, indépendamment de la taille du modèle. Ces quatre définitions de l'efficacité sont complémentaires plutôt que concurrentes, et se combinent en pratique dans les modèles ouverts actuels (par exemple Mistral 7B - Instruct applique à la fois une architecture efficace et un fine-tuning d'alignement).

## 4. Perspectives et réflexions

**Tendances qui se dégagent.** Les quatre articles marquent collectivement un changement de paradigme dans la recherche sur les LLM entre 2022 et 2023 : on passe d'une logique de "scaling" pur (plus de paramètres = mieux) à une logique d'optimisation multi-critères, où la donnée, l'architecture et l'alignement comptent autant que la taille brute. Cette évolution est visible dans la manière dont chaque article s'appuie explicitement sur les résultats du précédent : LLaMA applique directement les lois d'échelle de Chinchilla ; les modèles ouverts qui ont suivi Mistral 7B ont quasi-systématiquement adopté le combo GQA + SWA ; et le RLHF popularisé par InstructGPT est désormais une étape standard du pipeline d'entraînement de tout modèle destiné au grand public.

**Approches les plus prometteuses.** La combinaison de plusieurs leviers simultanément semble la direction la plus fructueuse : un modèle compact, entraîné selon un ratio données/taille proche de l'optimum de Chinchilla, doté d'une architecture d'inférence efficace comme celle de Mistral, et affiné par retour humain comme InstructGPT, cumule des gains qui semblent largement additifs plutôt que redondants. C'est exactement la recette suivie par la quasi-totalité des modèles ouverts publiés depuis 2023 (Mistral, Llama 2/3, Gemma, Qwen, etc.).

**Limites et difficultés communes.** Deux difficultés reviennent systématiquement dans ces travaux. D'abord, la reproductibilité reste partielle : même quand les poids sont publiés (LLaMA, Mistral), les données d'entraînement précises et les recettes de filtrage restent rarement détaillées intégralement, et pour InstructGPT, les données de préférence humaine elles-mêmes ne sont pas publiques. Ensuite, chaque levier d'amélioration a un coût caché : les lois d'échelle de Chinchilla nécessitent d'entraîner des centaines de modèles pour être établies empiriquement, l'ouverture des données de LLaMA ne garantit pas leur qualité ou leur absence de biais, et le RLHF d'InstructGPT dépend d'un travail d'annotation humaine coûteux, difficile à faire évoluer et potentiellement biaisé par les préférences des annotateurs recrutés.

**Orientations futures.** Ces limites suggèrent plusieurs directions de recherche actives : réduire la dépendance au RLHF via des méthodes moins coûteuses en supervision humaine (par exemple l'apprentissage à partir de retours générés par IA, ou l'optimisation directe des préférences) ; mieux caractériser et documenter les données d'entraînement pour améliorer la reproductibilité et l'audit des modèles ; et continuer à explorer des architectures efficaces (attention creuse, mélanges d'experts) pour repousser encore le compromis entre taille, coût d'inférence et qualité.

## 5. Conclusion

Cette méta-analyse de quatre articles fondateurs - Chinchilla, LLaMA, Mistral 7B et InstructGPT - montre que l'amélioration des grands modèles de langage entre 2022 et 2023 ne s'est pas jouée uniquement sur le terrain de la taille, mais sur celui de l'efficacité au sens large : efficacité de l'allocation du calcul et des données (Chinchilla, LLaMA), efficacité architecturale à l'inférence (Mistral 7B), et efficacité de l'alignement comportemental (InstructGPT). Chaque article répond à une limite concrète de l'approche "plus gros est meilleur" - coût de calcul, dépendance aux données propriétaires, coût d'inférence, et décalage entre objectif d'entraînement et besoin réel de l'utilisateur.

Le champ a depuis clairement intégré ces quatre leçons : les modèles ouverts actuels combinent systématiquement un ratio données/paramètres proche de l'optimum de Chinchilla, des architectures d'attention efficaces héritées de travaux comme Mistral, et une étape d'alignement par retour humain ou automatisé inspirée d'InstructGPT. La prochaine étape probable du domaine consiste à rendre ces trois leviers eux-mêmes plus efficaces et plus reproductibles, notamment en réduisant le coût humain de l'alignement et en améliorant la transparence des données d'entraînement.

## Références

1. Hoffmann, J. et al. (2022). *Training Compute-Optimal Large Language Models*. NeurIPS 2022 ; arXiv:2203.15556. https://arxiv.org/abs/2203.15556
2. Touvron, H. et al. (2023). *LLaMA: Open and Efficient Foundation Language Models*. arXiv:2302.13971. https://arxiv.org/abs/2302.13971
3. Jiang, A. Q. et al. (2023). *Mistral 7B*. arXiv:2310.06825. https://arxiv.org/abs/2310.06825
4. Ouyang, L. et al. (2022). *Training Language Models to Follow Instructions with Human Feedback*. NeurIPS 2022 ; arXiv:2203.02155. https://arxiv.org/abs/2203.02155